# Grupa A – Regresja liniowa i logistyczna

Sprawdźcie, dlaczego zwykła regresja liniowa nie nadaje się do pytania „tak czy nie?”, jak działa regresja logistyczna (która mimo nazwy jest **modelem klasyfikacyjnym**) i jak wybór progu decyzyjnego zmienia wyniki modelu oraz zysk banku.

`y = yes` oznacza założenie lokaty. `duration` jest wyłączone, bo przed rozmową nie jest znane.

In [ ]:
# Uruchom tylko w Google Colab; lokalnie w VS Code pomiń tę komórkę.
!git clone https://github.com/poochacz/Podstawy_technologii_ML.git /content/eda_ml_workshop_repo

In [ ]:
import sys
from pathlib import Path

COLAB_ROOT = Path("/content/eda_ml_workshop_repo/02_ml_bank_marketing_colab")
current = Path.cwd().resolve()
candidates = [COLAB_ROOT]
for directory in (current, *current.parents):
    candidates.extend((directory / "02_ml_bank_marketing_colab", directory))

ROOT = next(
    (candidate.resolve() for candidate in candidates if (candidate / "scripts").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Nie znaleziono folderu 02_ml_bank_marketing_colab.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"
print("Środowisko gotowe:", ROOT)

In [ ]:
from scripts.data_setup import prepare_model_data

N_RECORDS = 2000
RANDOM_STATE = 42
df, data_metadata = prepare_model_data(
    n=N_RECORDS,
    random_state=RANDOM_STATE,
    data_dir=DATA_DIR,
)
print(f"Dane gotowe: {df.shape[0]} rekordów, {df.shape[1] - 1} cech.")

In [ ]:
from scripts.modeling import preprocess_data

df_for_modelling = preprocess_data(df)

## 1. Train/test split

Dzielimy 2000 klientów na 1600 do treningu i 400 do testu. Test jest zawsze ten sam, dzięki czemu możemy uczciwie porównywać modele.
Accuracy modelu naiwnego (który zawsze odpowiada „brak lokaty”) to 88,8%.

In [ ]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 400
SPLIT_SEED = 42

# usuwamy y (czyli 'target') - dlaczego?
X = df_for_modelling.drop(columns="y")
y = df_for_modelling["y"]

# Tutaj budujemy dane do treningu i testy. X to cechy, a y to target - czyli wzięcie lub nie lokaty
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=SPLIT_SEED,
    stratify=y,
)

print(f"Train: {len(X_train)} rekordów; test: {len(X_test)} rekordów.")

In [ ]:
# można zobaczyć jakie cechy są wykorzystywane przez model - wiele jest dodanych tylko na potrzeby trenowania modelu
X_train.columns.tolist()

## 2. Regresja liniowa

Regresja liniowa przewiduje **liczbę**, np. cenę mieszkania albo wynagrodzenie. Spróbujmy użyć jej do naszego pytania: zamieniamy `yes` na 1, a `no` na 0. Wynik bliski 1 powinien oznaczać „pewnie założy lokatę”, a bliski 0 – „raczej nie”.

In [ ]:
from sklearn.linear_model import LinearRegression

# Regresja liniowa potrzebuje liczb, więc zamieniamy yes/no na 1/0
y_train_number = (y_train == "yes").astype(int)

linear = LinearRegression()
linear.fit(X_train, y_train_number)

linear_predictions = linear.predict(X_test)
print(f"Najmniejsza przewidziana wartość: {linear_predictions.min():.2f}")
print(f"Największa przewidziana wartość: {linear_predictions.max():.2f}")
print(f"Wyników poniżej 0: {(linear_predictions < 0).sum()} z {len(linear_predictions)}")

Co znaczy szansa na lokatę równa −0,05? Nic – szansa nie może być ujemna ani większa niż 1. Regresja liniowa nie wie, że odpowiedź może być tylko 0 albo 1: dopasowuje prostą zależność, która może wyjść poza ten zakres. Potrzebujemy modelu stworzonego do klasyfikacji.

## 3. Regresja logistyczna – model klasyfikacyjny

> **Uwaga: regresja logistyczna to model KLASYFIKACYJNY, a nie regresyjny.** Nazwa jest historyczna i myląca. Model przewiduje klasę (`yes` / `no`), a nie liczbę taką jak cena czy wynagrodzenie.

Model działa w dwóch krokach:
1. Liczy **prawdopodobieństwo** `yes` – zawsze liczbę od 0 do 1. W środku ma wzór podobny do regresji liniowej, ale jego wynik jest „ściskany” do przedziału 0–1 przez funkcję logistyczną (stąd nazwa).
2. Porównuje prawdopodobieństwo z **progiem** (domyślnie 0,5): powyżej progu odpowiada `yes`, poniżej `no`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# max_iter = ile kroków model może wykonać, szukając najlepszych współczynników
logistic = LogisticRegression(max_iter=1000)
logistic.fit(X_train, y_train)  # y zostaje jako yes/no – to klasyfikacja

# Krok 1: prawdopodobieństwo yes (kolumna 1, bo logistic.classes_ to ['no', 'yes'])
logistic_probabilities = logistic.predict_proba(X_test)[:, 1]
# Krok 2: klasa yes/no przy domyślnym progu 0,5
y_pred_logistic = logistic.predict(X_test)

print(f"Prawdopodobieństwa od {logistic_probabilities.min():.2f} do {logistic_probabilities.max():.2f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_logistic):.1%} (baseline: {accuracy_score(y_test, ['no'] * len(y_test)):.1%})")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, y_pred_logistic):.1%} (baseline: 50%)")

### 3a. Porównanie obu modeli

Ten sam test, te same cechy. Po lewej wyniki regresji liniowej, po prawej prawdopodobieństwa z regresji logistycznej.

In [ ]:
from scripts.modeling import plot_regression_predictions

plot_regression_predictions(linear_predictions, logistic_probabilities)

## 4. Jak model myśli?

Regresja logistyczna przypisuje każdej cesze **współczynnik**. Dodatni zwiększa szansę na `yes`, ujemny ją zmniejsza, a im dalej od zera, tym silniejszy wpływ. W przeciwieństwie do lasu losowego taki model można czytać wprost.

Dwie uwagi:
- dla `age`, `campaign` i `pdays` współczynnik dotyczy jednego roku, jednego kontaktu albo jednego dnia, więc nie porównujcie go wprost z kolumnami 0/1,
- współczynnik pokazuje związek, a nie przyczynę.

In [ ]:
from scripts.explainability import plot_logistic_coefficients

plot_logistic_coefficients(logistic, top_n=12)

## 5. Próg decyzyjny

Domyślny próg 0,5 nie jest żadną świętością. Możemy powiedzieć „dzwonimy do klienta, gdy szansa na lokatę wynosi co najmniej 20%”. Niższy próg oznacza więcej odpowiedzi `yes`: znajdziemy więcej chętnych, ale też zadzwonimy do większej liczby osób, które odmówią.

Na 2000 klientach test ma tylko 45 chętnych, więc tablica pomyłek skakałaby o pojedyncze osoby. Dlatego tutaj wczytujemy **cały zbiór (ponad 41 tys. klientów)** i dzielimy go na trening (80%) i test (20%).

In [ ]:
df_full, _ = prepare_model_data(n=None, random_state=RANDOM_STATE, data_dir=DATA_DIR)
df_full_for_modelling = preprocess_data(df_full)

X_full = df_full_for_modelling.drop(columns="y")
y_full = df_full_for_modelling["y"]

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=SPLIT_SEED,
    stratify=y_full,
)

logistic_full = LogisticRegression(max_iter=1000)
logistic_full.fit(X_train_full, y_train_full)
probabilities_full = logistic_full.predict_proba(X_test_full)[:, 1]

print(f"Train: {len(X_train_full)} rekordów; test: {len(X_test_full)} rekordów.")

### 5a. Wybierzcie próg

Zmieniajcie `THRESHOLD` (np. 0,5; 0,3; 0,2; 0,1) i obserwujcie tablicę pomyłek.

- **Recall** – jaki procent chętnych znaleźliśmy.
- **Precision** – jaki procent klientów, którym model powiedział `yes`, naprawdę założył lokatę.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score

# Zmień próg: od jakiej szansy model mówi "yes"
THRESHOLD = 0.5

y_pred_threshold = np.where(probabilities_full >= THRESHOLD, "yes", "no")

print(f"Accuracy: {accuracy_score(y_test_full, y_pred_threshold):.1%}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test_full, y_pred_threshold):.1%}")
print(f"Recall: {recall_score(y_test_full, y_pred_threshold, pos_label='yes'):.1%}")
print(f"Precision: {precision_score(y_test_full, y_pred_threshold, pos_label='yes', zero_division=0):.1%}")

pd.crosstab(y_test_full, y_pred_threshold, rownames=["naprawdę"], colnames=["przewidziane"])

### 5b. Wszystkie progi naraz

Wykres pokazuje, jak zmieniają się cztery miary, gdy przesuwamy próg od 0,02 do 0,95.

W grupach B i C do znajdowania chętnych służyło `class_weight="balanced"`. Obniżenie progu daje podobny efekt: oba sposoby sprawiają, że model częściej mówi `yes`.

In [ ]:
from scripts.modeling import plot_threshold_metrics

plot_threshold_metrics(y_test_full, probabilities_full)

## 6. Próg z punktu widzenia banku

Która miara jest najważniejsza? To zależy od biznesu. Załóżmy, że:
- każdy telefon kosztuje bank `CALL_COST` zł (czas konsultanta),
- każda założona lokata przynosi `DEPOSIT_PROFIT` zł zysku.

Bank dzwoni tylko do klientów, których szansa na lokatę jest co najmniej równa progowi. Dla każdego progu liczymy zysk = lokaty × `DEPOSIT_PROFIT` − telefony × `CALL_COST` i szukamy progu z największym zyskiem.

Zmieniajcie koszty i zysk i obserwujcie, jak przesuwa się najlepszy próg. Podpowiedź: dzwonienie opłaca się, gdy szansa × zysk z lokaty > koszt telefonu.

In [ ]:
from scripts.modeling import plot_profit_by_threshold

CALL_COST = 10        # koszt jednego telefonu w zł
DEPOSIT_PROFIT = 100  # zysk z jednej lokaty w zł

plot_profit_by_threshold(y_test_full, probabilities_full, CALL_COST, DEPOSIT_PROFIT)

## Pytania do dyskusji

1. Dlaczego regresja liniowa nie nadaje się do przewidywania odpowiedzi „tak / nie”?
2. Obie metody mają w nazwie „regresja”. Czym różni się regresja logistyczna od liniowej i dlaczego jest modelem klasyfikacyjnym?
3. Które cechy najbardziej zwiększają, a które zmniejszają szansę na lokatę? Czy to znaczy, że je powodują?
4. Jak obniżanie progu zmienia accuracy, recall i precision? Dlaczego nie istnieje jeden najlepszy próg dla wszystkich?
5. Jak zmienia się najlepszy próg, gdy telefon drożeje albo lokata przynosi mniej zysku? Kiedy opłaca się zadzwonić do wszystkich?